# 📄 Application de Facturation avec Signature UOV

**École Nationale Supérieure Polytechnique de Yaoundé**  
Science de l'Information - Cryptographie  
Année Académique : 2025/2026

---

## 🔐 Qu'est-ce que UOV ?

Le schéma **UOV (Unbalanced Oil and Vinegar)** est un système de signature numérique basé sur la cryptographie multivariée, conçu pour **résister aux attaques quantiques**.

### Principe de Fonctionnement

- **Variables Oil (o = 5)** : Ne peuvent pas se multiplier entre elles
- **Variables Vinegar (v = 10)** : Peuvent se multiplier entre elles et avec Oil
- **Déséquilibre** : v > o (d'où "Unbalanced")
- **Corps fini** : GF(31)

### Architecture

Ce notebook implémente une application complète de facturation avec:
- ✅ Gestion des clients
- ✅ Création de factures
- ✅ Signature cryptographique UOV
- ✅ Vérification des signatures
- ✅ Interface Gradio interactive

---

## 📦 Installation des Dépendances

Exécutez cette cellule pour installer tous les packages nécessaires.

In [ ]:
# Installation des dépendances
!pip install -q gradio
!pip install -q sqlalchemy
!pip install -q pandas

print("✅ Installation terminée!")

## 🔧 Imports et Configuration

In [ ]:
import gradio as gr
import hashlib
import json
import secrets
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Tuple, List, Optional
from sqlalchemy import create_engine, Column, Integer, String, Float, Boolean, DateTime, ForeignKey, LargeBinary
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship

print("✅ Tous les modules importés avec succès!")

## 🎓 Classe UOVSigner - Implémentation de la Signature UOV

Cette classe implémente le schéma de signature UOV avec:
1. **Génération des clés** : Création de la paire clé publique/privée
2. **Signature** : Processus hash-and-sign
3. **Vérification** : Validation de l'authenticité

In [ ]:
class UOVSigner:
    """
    Implémentation simplifiée du schéma de signature UOV
    
    Paramètres:
    - q: taille du corps fini (nombre premier) = 31
    - o: nombre de variables Oil = 5
    - v: nombre de variables Vinegar = 10
    - n = o + v: nombre total de variables = 15
    """
    
    def __init__(self, q=31, o=5, v=10):
        self.q = q  # Corps fini GF(q)
        self.o = o  # Variables Oil
        self.v = v  # Variables Vinegar
        self.n = o + v  # Total variables
        
        if v <= o:
            raise ValueError("UOV nécessite v > o (Unbalanced)")
    
    def generate_keys(self) -> Tuple[dict, dict]:
        """Génère une paire de clés (publique, privée) pour UOV"""
        # Génération de l'application centrale F
        F_coefficients = self._generate_central_map()
        
        # Génération de la transformation affine T (inversible)
        T_matrix, T_vector, T_inv_matrix, T_inv_vector = self._generate_affine_transform()
        
        # Clé privée: (F, T, T_inv)
        private_key = {
            'F_coefficients': F_coefficients,
            'T_matrix': T_matrix,
            'T_vector': T_vector,
            'T_inv_matrix': T_inv_matrix,
            'T_inv_vector': T_inv_vector,
            'q': self.q,
            'o': self.o,
            'v': self.v
        }
        
        # Clé publique: P = F ∘ T
        P_coefficients = self._compose_maps(F_coefficients, T_matrix, T_vector)
        
        public_key = {
            'P_coefficients': P_coefficients,
            'q': self.q,
            'o': self.o,
            'v': self.v,
            'n': self.n
        }
        
        return public_key, private_key
    
    def _generate_central_map(self) -> List[dict]:
        """Génère l'application centrale F avec structure Oil-Vinegar"""
        F_coefficients = []
        
        for k in range(self.o):
            poly = {
                'quadratic': {},  # Termes quadratiques
                'linear': {},     # Termes linéaires
                'constant': 0     # Constante
            }
            
            # Termes quadratiques: seulement Vinegar-Vinegar et Vinegar-Oil
            # PAS de Oil-Oil (c'est la clé de UOV!)
            
            # Vinegar-Vinegar
            for i in range(self.v):
                for j in range(i, self.v):
                    coef = secrets.randbelow(self.q)
                    if coef != 0:
                        poly['quadratic'][(i, j)] = coef
            
            # Vinegar-Oil
            for i in range(self.v):
                for j in range(self.v, self.n):
                    coef = secrets.randbelow(self.q)
                    if coef != 0:
                        poly['quadratic'][(i, j)] = coef
            
            # Termes linéaires
            for i in range(self.n):
                coef = secrets.randbelow(self.q)
                if coef != 0:
                    poly['linear'][i] = coef
            
            poly['constant'] = secrets.randbelow(self.q)
            F_coefficients.append(poly)
        
        return F_coefficients
    
    def _generate_affine_transform(self) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Génère une transformation affine inversible T et son inverse T^-1"""
        # Génère une matrice inversible M
        while True:
            M = np.random.randint(0, self.q, (self.n, self.n))
            det = int(round(np.linalg.det(M))) % self.q
            if det != 0:
                break
        
        c = np.random.randint(0, self.q, self.n)
        M_inv = self._matrix_inverse_mod(M, self.q)
        c_inv = (-M_inv @ c) % self.q
        
        return M, c, M_inv, c_inv
    
    def _matrix_inverse_mod(self, M: np.ndarray, mod: int) -> np.ndarray:
        """Calcule l'inverse d'une matrice modulo mod (version simplifiée)"""
        try:
            M_inv = np.linalg.inv(M)
            M_inv = np.round(M_inv).astype(int) % mod
            return M_inv
        except:
            return np.eye(self.n, dtype=int)
    
    def _compose_maps(self, F_coefficients: List[dict], T_matrix: np.ndarray, T_vector: np.ndarray) -> List[dict]:
        """Compose F avec T pour obtenir P = F ∘ T"""
        return F_coefficients  # Version simplifiée
    
    def sign(self, message: str, private_key: dict) -> bytes:
        """Signe un message avec UOV"""
        # 1. Hacher le message
        hash_value = self._hash_message(message, private_key['o'])
        
        # 2. Résoudre F(U) = hash_value
        vinegar_vars = [secrets.randbelow(private_key['q']) for _ in range(private_key['v'])]
        oil_vars = self._solve_oil_variables(
            hash_value,
            vinegar_vars,
            private_key['F_coefficients'],
            private_key['q']
        )
        
        U = vinegar_vars + oil_vars
        
        # 3. Calculer s = T^-1(U)
        U_array = np.array(U)
        s_array = (private_key['T_inv_matrix'] @ U_array + private_key['T_inv_vector']) % private_key['q']
        s = s_array.tolist()
        
        signature_data = {
            'signature': s,
            'q': private_key['q'],
            'o': private_key['o'],
            'v': private_key['v']
        }
        
        return json.dumps(signature_data).encode()
    
    def _hash_message(self, message: str, output_length: int) -> List[int]:
        """Hache un message pour obtenir un vecteur dans GF(q)^o"""
        hash_obj = hashlib.sha256(message.encode())
        hash_bytes = hash_obj.digest()
        
        hash_vector = []
        for i in range(output_length):
            byte_val = hash_bytes[i % len(hash_bytes)]
            hash_vector.append(byte_val % self.q)
        
        return hash_vector
    
    def _solve_oil_variables(self, target: List[int], vinegar_vars: List[int],
                           F_coefficients: List[dict], q: int) -> List[int]:
        """Résout les variables Oil étant donné les variables Vinegar"""
        v = len(vinegar_vars)
        o = len(target)
        oil_vars = [0] * o
        
        for k in range(o):
            poly = F_coefficients[k]
            known_part = poly['constant']
            
            # Termes quadratiques Vinegar-Vinegar
            for (i, j), coef in poly['quadratic'].items():
                if i < v and j < v:
                    known_part += coef * vinegar_vars[i] * vinegar_vars[j]
            
            # Termes linéaires Vinegar
            for i, coef in poly['linear'].items():
                if i < v:
                    known_part += coef * vinegar_vars[i]
            
            known_part %= q
            oil_vars[k] = (target[k] - known_part) % q
        
        return oil_vars
    
    def verify(self, message: str, signature: bytes, public_key: dict) -> bool:
        """Vérifie une signature UOV"""
        try:
            sig_data = json.loads(signature.decode())
            s = sig_data['signature']
            q = sig_data['q']
            o = sig_data['o']
            
            hash_value = self._hash_message(message, o)
            P_s = self._evaluate_polynomial(s, public_key['P_coefficients'], q)
            
            return P_s == hash_value
        except Exception as e:
            print(f"Erreur de vérification: {e}")
            return False
    
    def _evaluate_polynomial(self, x: List[int], P_coefficients: List[dict], q: int) -> List[int]:
        """Évalue les polynômes P en x"""
        result = []
        
        for poly in P_coefficients:
            value = poly['constant']
            
            for (i, j), coef in poly['quadratic'].items():
                value += coef * x[i] * x[j]
            
            for i, coef in poly['linear'].items():
                value += coef * x[i]
            
            value %= q
            result.append(value)
        
        return result


class NumpyEncoder(json.JSONEncoder):
    """Encodeur JSON pour les arrays numpy"""
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.integer):
            return int(obj)
        return super().default(obj)


print("✅ Classe UOVSigner chargée avec succès!")

## 🗄️ Configuration de la Base de Données SQLite

Cette section configure la base de données pour stocker:
- **Clients** : informations de contact
- **Factures** : montants, signatures UOV, clés cryptographiques

In [ ]:
# Configuration SQLAlchemy
Base = declarative_base()

class Client(Base):
    """Modèle pour les clients"""
    __tablename__ = "clients"
    
    id = Column(Integer, primary_key=True, index=True)
    nom = Column(String, nullable=False)
    email = Column(String, unique=True, nullable=False)
    adresse = Column(String)
    telephone = Column(String)
    date_creation = Column(DateTime, default=datetime.utcnow)
    
    factures = relationship("Facture", back_populates="client")


class Facture(Base):
    """Modèle pour les factures"""
    __tablename__ = "factures"
    
    id = Column(Integer, primary_key=True, index=True)
    client_id = Column(Integer, ForeignKey("clients.id"), nullable=False)
    numero_facture = Column(String, unique=True, nullable=False)
    date_emission = Column(DateTime, default=datetime.utcnow)
    montant_total = Column(Float, nullable=False)
    description = Column(String)
    items = Column(String)  # JSON string
    
    # Champs de signature UOV
    signature_uov = Column(LargeBinary, nullable=True)
    cle_publique = Column(LargeBinary, nullable=True)
    cle_privee = Column(LargeBinary, nullable=True)
    est_signee = Column(Boolean, default=False)
    date_signature = Column(DateTime, nullable=True)
    
    date_creation = Column(DateTime, default=datetime.utcnow)
    
    client = relationship("Client", back_populates="factures")


# Initialisation de la base de données
DATABASE_URL = "sqlite:///billing_uov.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

# Créer les tables
Base.metadata.create_all(bind=engine)

print("✅ Base de données SQLite initialisée: billing_uov.db")

## 🛠️ Fonctions Utilitaires pour l'Application

In [ ]:
def generer_numero_facture():
    """Génère un numéro de facture unique"""
    year = datetime.now().year
    db = SessionLocal()
    count = db.query(Facture).count()
    db.close()
    return f"FACT-{year}-{count+1:05d}"


def generer_cles_uov():
    """Génère une paire de clés UOV"""
    signer = UOVSigner(q=31, o=5, v=10)
    public_key, private_key = signer.generate_keys()
    
    public_key_bytes = json.dumps(public_key, cls=NumpyEncoder).encode()
    private_key_bytes = json.dumps(private_key, cls=NumpyEncoder).encode()
    
    return public_key_bytes, private_key_bytes


def signer_message(message: str, private_key_bytes: bytes) -> bytes:
    """Signe un message avec UOV"""
    private_key = json.loads(private_key_bytes.decode())
    
    # Reconstruire les arrays numpy
    private_key['T_matrix'] = np.array(private_key['T_matrix'])
    private_key['T_vector'] = np.array(private_key['T_vector'])
    private_key['T_inv_matrix'] = np.array(private_key['T_inv_matrix'])
    private_key['T_inv_vector'] = np.array(private_key['T_inv_vector'])
    
    signer = UOVSigner(q=private_key['q'], o=private_key['o'], v=private_key['v'])
    return signer.sign(message, private_key)


def verifier_signature(message: str, signature: bytes, public_key_bytes: bytes) -> bool:
    """Vérifie une signature UOV"""
    public_key = json.loads(public_key_bytes.decode())
    signer = UOVSigner(q=public_key['q'], o=public_key['o'], v=public_key['v'])
    return signer.verify(message, signature, public_key)


print("✅ Fonctions utilitaires chargées!")

## 📊 Fonctions de Gestion des Clients

In [ ]:
def creer_client(nom: str, email: str, adresse: str = "", telephone: str = ""):
    """Crée un nouveau client"""
    db = SessionLocal()
    try:
        # Vérifier si l'email existe déjà
        existing = db.query(Client).filter(Client.email == email).first()
        if existing:
            return "❌ Erreur: Cet email existe déjà!"
        
        client = Client(
            nom=nom,
            email=email,
            adresse=adresse if adresse else None,
            telephone=telephone if telephone else None
        )
        db.add(client)
        db.commit()
        db.refresh(client)
        return f"✅ Client créé avec succès! ID: {client.id}"
    except Exception as e:
        db.rollback()
        return f"❌ Erreur: {str(e)}"
    finally:
        db.close()


def lister_clients():
    """Liste tous les clients"""
    db = SessionLocal()
    try:
        clients = db.query(Client).all()
        if not clients:
            return pd.DataFrame(columns=["ID", "Nom", "Email", "Téléphone", "Date Création"])
        
        data = []
        for c in clients:
            data.append({
                "ID": c.id,
                "Nom": c.nom,
                "Email": c.email,
                "Téléphone": c.telephone or "-",
                "Date Création": c.date_creation.strftime("%Y-%m-%d %H:%M")
            })
        return pd.DataFrame(data)
    finally:
        db.close()


print("✅ Fonctions clients chargées!")

## 📄 Fonctions de Gestion des Factures

In [ ]:
def creer_facture(client_id: int, description: str, items_json: str):
    """Crée une nouvelle facture avec génération automatique des clés UOV"""
    db = SessionLocal()
    try:
        # Vérifier que le client existe
        client = db.query(Client).filter(Client.id == client_id).first()
        if not client:
            return "❌ Erreur: Client introuvable!"
        
        # Parser les items
        try:
            items = json.loads(items_json)
            montant_total = sum(item['quantite'] * item['prix_unitaire'] for item in items)
        except:
            return "❌ Erreur: Format des items invalide! Utilisez le format JSON."
        
        # Générer les clés UOV
        public_key, private_key = generer_cles_uov()
        
        # Créer la facture
        facture = Facture(
            client_id=client_id,
            numero_facture=generer_numero_facture(),
            description=description,
            items=items_json,
            montant_total=montant_total,
            cle_publique=public_key,
            cle_privee=private_key,
            est_signee=False
        )
        
        db.add(facture)
        db.commit()
        db.refresh(facture)
        
        return f"✅ Facture {facture.numero_facture} créée! Montant: {montant_total:.2f} FCFA"
    except Exception as e:
        db.rollback()
        return f"❌ Erreur: {str(e)}"
    finally:
        db.close()


def lister_factures():
    """Liste toutes les factures"""
    db = SessionLocal()
    try:
        factures = db.query(Facture).join(Client).all()
        if not factures:
            return pd.DataFrame(columns=["ID", "N° Facture", "Client", "Montant", "Signée", "Date"])
        
        data = []
        for f in factures:
            data.append({
                "ID": f.id,
                "N° Facture": f.numero_facture,
                "Client": f.client.nom,
                "Montant": f"{f.montant_total:.2f} FCFA",
                "Signée": "✅ Oui" if f.est_signee else "❌ Non",
                "Date": f.date_emission.strftime("%Y-%m-%d %H:%M")
            })
        return pd.DataFrame(data)
    finally:
        db.close()


def signer_facture(facture_id: int):
    """Signe une facture avec UOV"""
    db = SessionLocal()
    try:
        facture = db.query(Facture).filter(Facture.id == facture_id).first()
        if not facture:
            return "❌ Facture introuvable!"
        
        if facture.est_signee:
            return "⚠️ Cette facture est déjà signée!"
        
        # Créer le message à signer
        message = f"{facture.numero_facture}|{facture.client_id}|{facture.montant_total}|{facture.items}"
        
        # Signer avec UOV
        signature = signer_message(message, facture.cle_privee)
        
        # Sauvegarder la signature
        facture.signature_uov = signature
        facture.est_signee = True
        facture.date_signature = datetime.utcnow()
        
        db.commit()
        return f"✅ Facture {facture.numero_facture} signée avec succès!"
    except Exception as e:
        db.rollback()
        return f"❌ Erreur: {str(e)}"
    finally:
        db.close()


def verifier_facture(facture_id: int):
    """Vérifie la signature d'une facture"""
    db = SessionLocal()
    try:
        facture = db.query(Facture).filter(Facture.id == facture_id).first()
        if not facture:
            return "❌ Facture introuvable!"
        
        if not facture.est_signee:
            return "⚠️ Cette facture n'est pas encore signée!"
        
        # Recréer le message
        message = f"{facture.numero_facture}|{facture.client_id}|{facture.montant_total}|{facture.items}"
        
        # Vérifier la signature
        is_valid = verifier_signature(message, facture.signature_uov, facture.cle_publique)
        
        if is_valid:
            return f"✅ Signature VALIDE pour {facture.numero_facture}! La facture est authentique."
        else:
            return f"❌ Signature INVALIDE pour {facture.numero_facture}! La facture a été modifiée."
    finally:
        db.close()


print("✅ Fonctions factures chargées!")

## 🎨 Interface Gradio - Application Interactive

**🚀 Cette cellule lance l'interface Gradio**

L'interface s'affichera directement dans le notebook (pas besoin de lien externe).

In [ ]:
# Créer l'interface Gradio
with gr.Blocks(title="Facturation UOV", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 📄 Application de Facturation avec Signature UOV
        
        **École Nationale Supérieure Polytechnique de Yaoundé**
        
        Application de facturation sécurisée avec signature post-quantique UOV (Unbalanced Oil and Vinegar)
        
        ---
        """
    )
    
    with gr.Tabs():
        # ========== ONGLET 1: CLIENTS ==========
        with gr.Tab("👥 Gestion des Clients"):
            gr.Markdown("### Créer un Nouveau Client")
            
            with gr.Row():
                with gr.Column():
                    client_nom = gr.Textbox(label="Nom *", placeholder="Jean Dupont")
                    client_email = gr.Textbox(label="Email *", placeholder="jean.dupont@example.com")
                with gr.Column():
                    client_adresse = gr.Textbox(label="Adresse", placeholder="123 Rue Example")
                    client_telephone = gr.Textbox(label="Téléphone", placeholder="+237 6XX XX XX XX")
            
            btn_creer_client = gr.Button("➕ Créer Client", variant="primary")
            output_client = gr.Textbox(label="Résultat", interactive=False)
            
            gr.Markdown("### Liste des Clients")
            btn_refresh_clients = gr.Button("🔄 Actualiser")
            table_clients = gr.Dataframe(
                headers=["ID", "Nom", "Email", "Téléphone", "Date Création"],
                interactive=False
            )
            
            # Actions
            btn_creer_client.click(
                fn=creer_client,
                inputs=[client_nom, client_email, client_adresse, client_telephone],
                outputs=output_client
            )
            btn_refresh_clients.click(fn=lister_clients, outputs=table_clients)
        
        # ========== ONGLET 2: FACTURES ==========
        with gr.Tab("📄 Gestion des Factures"):
            gr.Markdown("### Créer une Nouvelle Facture")
            
            facture_client_id = gr.Number(label="ID Client *", precision=0)
            facture_description = gr.Textbox(label="Description *", placeholder="Prestation de services")
            
            gr.Markdown(
                """
                **Format des items (JSON):**
                ```json
                [
                    {"description": "Consultation", "quantite": 2, "prix_unitaire": 50000},
                    {"description": "Formation", "quantite": 1, "prix_unitaire": 150000}
                ]
                ```
                """
            )
            
            facture_items = gr.Code(
                label="Items (JSON) *",
                language="json",
                value='[{"description": "Article 1", "quantite": 1, "prix_unitaire": 10000}]'
            )
            
            btn_creer_facture = gr.Button("➕ Créer Facture", variant="primary")
            output_facture = gr.Textbox(label="Résultat", interactive=False)
            
            gr.Markdown("### Liste des Factures")
            btn_refresh_factures = gr.Button("🔄 Actualiser")
            table_factures = gr.Dataframe(
                headers=["ID", "N° Facture", "Client", "Montant", "Signée", "Date"],
                interactive=False
            )
            
            # Actions
            btn_creer_facture.click(
                fn=creer_facture,
                inputs=[facture_client_id, facture_description, facture_items],
                outputs=output_facture
            )
            btn_refresh_factures.click(fn=lister_factures, outputs=table_factures)
        
        # ========== ONGLET 3: SIGNATURE UOV ==========
        with gr.Tab("🔐 Signature & Vérification UOV"):
            gr.Markdown(
                """
                ### Signature Cryptographique Post-Quantique
                
                Cette section permet de signer et vérifier les factures avec le schéma UOV.
                
                **Processus de signature:**
                1. Hash SHA-256 du message (données de la facture)
                2. Choix aléatoire des variables Vinegar
                3. Résolution pour les variables Oil
                4. Application de la transformation T⁻¹
                
                **Vérification:**
                - Calcul de P(s) avec la clé publique
                - Comparaison avec hash(message)
                """
            )
            
            with gr.Row():
                with gr.Column():
                    gr.Markdown("#### Signer une Facture")
                    sign_facture_id = gr.Number(label="ID Facture", precision=0)
                    btn_signer = gr.Button("✍️ Signer", variant="primary")
                    output_signer = gr.Textbox(label="Résultat", interactive=False)
                
                with gr.Column():
                    gr.Markdown("#### Vérifier une Signature")
                    verify_facture_id = gr.Number(label="ID Facture", precision=0)
                    btn_verifier = gr.Button("🔍 Vérifier", variant="primary")
                    output_verifier = gr.Textbox(label="Résultat", interactive=False)
            
            # Actions
            btn_signer.click(
                fn=signer_facture,
                inputs=sign_facture_id,
                outputs=output_signer
            )
            btn_verifier.click(
                fn=verifier_facture,
                inputs=verify_facture_id,
                outputs=output_verifier
            )
        
        # ========== ONGLET 4: DÉMONSTRATION UOV ==========
        with gr.Tab("🎓 Démonstration UOV"):
            gr.Markdown(
                """
                ### Comprendre le Schéma UOV
                
                Cette section permet de tester le processus de signature/vérification UOV directement.
                
                **Paramètres:**
                - Corps fini: GF(31)
                - Variables Oil (o): 5
                - Variables Vinegar (v): 10
                - Total variables (n): 15
                """
            )
            
            demo_message = gr.Textbox(
                label="Message à signer",
                placeholder="Entrez votre message...",
                value="Ceci est un test de signature UOV"
            )
            
            btn_demo_sign = gr.Button("🔐 Générer Clés + Signer + Vérifier", variant="primary")
            
            demo_output = gr.Textbox(label="Résultat Complet", lines=15, interactive=False)
            
            def demo_uov_complete(message):
                """Démonstration complète de UOV"""
                try:
                    # 1. Générer les clés
                    public_key, private_key = generer_cles_uov()
                    
                    # 2. Signer
                    signature = signer_message(message, private_key)
                    
                    # 3. Vérifier
                    is_valid = verifier_signature(message, signature, public_key)
                    
                    # 4. Formater la sortie
                    output = f"""
✅ DÉMONSTRATION COMPLÈTE UOV
================================

📝 Message original:
{message}

🔑 Clés générées:
- Clé publique: {len(public_key)} bytes
- Clé privée: {len(private_key)} bytes

✍️ Signature:
{signature.decode()[:200]}... (tronqué)

🔍 Vérification:
{'✅ SIGNATURE VALIDE!' if is_valid else '❌ SIGNATURE INVALIDE!'}

📊 Statistiques:
- Corps fini: GF(31)
- Variables Oil: 5
- Variables Vinegar: 10
- Total: 15 variables

🎓 Explication:
Le message a été haché avec SHA-256, puis signé avec le schéma UOV.
La vérification confirme que P(s) = hash(message), prouvant l'authenticité.
                    """
                    return output
                except Exception as e:
                    return f"❌ Erreur: {str(e)}"
            
            btn_demo_sign.click(
                fn=demo_uov_complete,
                inputs=demo_message,
                outputs=demo_output
            )
    
    gr.Markdown(
        """
        ---
        
        ### 📚 Références
        
        - [UOV Official Site](https://www.uovsig.org/)
        - [NIST Post-Quantum Cryptography](https://csrc.nist.gov/projects/post-quantum-cryptography)
        - Superviseur: Dr. Tale Kalachi
        
        **⚠️ Note:** Cette implémentation est éducative. Pour la production, utilisez des paramètres NIST recommandés.
        """
    )

# Lancer l'application - L'interface s'affiche DANS le notebook
demo.launch()

## 🎯 Instructions d'Utilisation

### ✅ Sur Google Colab (GRATUIT):
1. **Exécutez toutes les cellules** dans l'ordre : `Runtime > Run all`
2. **L'interface Gradio s'affichera automatiquement** dans le notebook (pas de lien externe nécessaire)
3. **Utilisez l'application** directement dans la cellule

### ✅ Sur Kaggle (GRATUIT):
1. **Uploadez ce notebook**
2. **Activez Internet** : `Settings > Internet > On`
3. **Exécutez toutes les cellules**
4. **L'interface apparaîtra** dans la dernière cellule

### 📝 Workflow Typique:
1. **Créer des clients** dans l'onglet "👥 Gestion des Clients"
2. **Créer des factures** dans l'onglet "📄 Gestion des Factures" (les clés UOV sont générées automatiquement)
3. **Signer les factures** dans l'onglet "🔐 Signature & Vérification UOV"
4. **Vérifier les signatures** pour confirmer l'authenticité
5. **Tester UOV** dans l'onglet "🎓 Démonstration UOV" pour comprendre le processus

### 💡 Exemple Rapide:

**Étape 1 - Créer un client:**
- Nom: `Jean Dupont`
- Email: `jean.dupont@example.com`
- Téléphone: `+237 600 00 00 00`
- Cliquez sur "Créer Client" puis "Actualiser"

**Étape 2 - Créer une facture:**
- ID Client: `1`
- Description: `Consultation en cryptographie`
- Items (déjà pré-rempli, vous pouvez modifier)
- Cliquez sur "Créer Facture" puis "Actualiser"

**Étape 3 - Signer:**
- ID Facture: `1`
- Cliquez sur "✍️ Signer"

**Étape 4 - Vérifier:**
- ID Facture: `1`
- Cliquez sur "🔍 Vérifier"

---

**Bon apprentissage de la cryptographie post-quantique ! 🔐**